<a href="https://colab.research.google.com/github/zombimann/Mathematical-video-animations-and-visualization/blob/main/Projectile_Motion_Visualization_2D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
"""
Projectile Motion — Educational Animation
==========================================
Author  : Mugambi Ndwiga
Instagram: @craftsandengineering
GitHub  : github.com/zombimann/Mathematical-video-animations-and-visualization

A production-quality animated explainer on projectile motion covering:
  • Time of Flight   T = 2u sinθ / g
  • Maximum Height   H = u² sin²θ / (2g)
  • Range            R = u² sin2θ / g
  • Trajectory       y = x tanθ − (g x²) / (2u² cos²θ)

Designed for general audiences with embedded expert-level telemetry.
"""

# ============================================================
# CELL 2 — MAIN ANIMATION
# ============================================================

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.animation import FFMpegWriter
from matplotlib.patches import FancyArrowPatch, Arc, FancyBboxPatch
from matplotlib.lines import Line2D
import matplotlib.gridspec as gridspec
import math
import os
from IPython.display import HTML
from base64 import b64encode
from google.colab import files

# ─────────────────────────────────────────────────────────────
# PARAMETRIC LAUNCH CONFIGURATION
# ─────────────────────────────────────────────────────────────
launch_angle_deg   = 55          # θ  — launch angle in degrees
initial_speed      = 22.0        # u  — initial speed (m/s)
gravity            = 9.81        # g  — gravitational acceleration (m/s²)
projectile_radius  = 0.35        # visual radius of projectile sphere
trail_length       = 80          # number of trail points to keep

# ─────────────────────────────────────────────────────────────
# VIDEO OUTPUT CONFIGURATION
# ─────────────────────────────────────────────────────────────
output_path        = "./projectile_motion.mp4"
video_fps          = 30
video_dpi          = 110         # keeps file < 10 MB
frame_width_in     = 14.4        # 16:9 at 110 dpi → 1584×891 px
frame_height_in    = 8.1

# ─────────────────────────────────────────────────────────────
# COLOR PALETTE  — "Obsidian & Ember"
# Deep near-black background, warm ember accents, cool steel
# secondary, electric teal highlights, soft gold telemetry.
# ─────────────────────────────────────────────────────────────
CLR_BG          = "#0D0F14"   # obsidian background
CLR_PANEL       = "#131720"   # card / panel fill
CLR_PANEL_EDGE  = "#252C3D"   # card border
CLR_TRAJECTORY  = "#E8623A"   # ember orange — main arc
CLR_VELOCITY    = "#4FC3F7"   # steel blue  — velocity arrows
CLR_HEIGHT      = "#69F0AE"   # mint green  — height telemetry
CLR_RANGE       = "#FFD54F"   # warm gold   — range telemetry
CLR_TIME        = "#CE93D8"   # soft violet — time telemetry
CLR_ACCEL       = "#FF6E6E"   # coral red   — gravity arrow
CLR_ANGLE_ARC   = "#FFF176"   # lemon       — angle arc
CLR_TEXT_HI     = "#F5F5F5"   # high-contrast white text
CLR_TEXT_MID    = "#A8B2C8"   # mid-contrast label text
CLR_TEXT_DIM    = "#4E576B"   # dim text
CLR_WATERMARK   = "#FFFFFF"   # watermark (set alpha separately)
CLR_GRID        = "#1C2230"   # subtle grid lines
CLR_CLOSING_BG  = "#0D0F14"   # closing card background

FONT_TITLE      = "DejaVu Sans"
FONT_BODY       = "DejaVu Sans"
FONT_MATH       = "DejaVu Sans"

# ─────────────────────────────────────────────────────────────
# DERIVED PHYSICS QUANTITIES
# ─────────────────────────────────────────────────────────────
theta_rad    = math.radians(launch_angle_deg)
sin_theta    = math.sin(theta_rad)
cos_theta    = math.cos(theta_rad)
tan_theta    = math.tan(theta_rad)

time_of_flight = 2 * initial_speed * sin_theta / gravity          # T
max_height     = (initial_speed**2 * sin_theta**2) / (2*gravity)  # H
range_total    = (initial_speed**2 * math.sin(2*theta_rad)) / gravity  # R

def x_pos(t):
    return initial_speed * cos_theta * t

def y_pos(t):
    return initial_speed * sin_theta * t - 0.5 * gravity * t**2

def vx(t):
    return initial_speed * cos_theta

def vy(t):
    return initial_speed * sin_theta - gravity * t

def speed(t):
    return math.hypot(vx(t), vy(t))

def trajectory_y(x):
    return x * tan_theta - (gravity * x**2) / (2 * initial_speed**2 * cos_theta**2)

# World coordinate limits with 15% padding
PAD_X = range_total * 0.12
PAD_Y = max_height  * 0.22
WORLD_X = (-PAD_X, range_total + PAD_X)
WORLD_Y = (-max_height * 0.12, max_height + PAD_Y)

# Pre-compute full trajectory curve
t_dense = np.linspace(0, time_of_flight, 500)
traj_x  = x_pos(t_dense)
traj_y  = y_pos(t_dense)

# ─────────────────────────────────────────────────────────────
# ANIMATION TIMELINE  (frames at 30 fps)
# Scene 1 : Hook title card                   0  – 60   ( 2 s)
# Scene 2 : Axes + angle reveal               60 – 120  ( 2 s)
# Scene 3 : Projectile launch & flight       120 – 330  ( 7 s)
# Scene 4 : Height annotation               330 – 390  ( 2 s)
# Scene 5 : Range annotation                390 – 450  ( 2 s)
# Scene 6 : Time-of-flight card             450 – 510  ( 2 s)
# Scene 7 : Equation of trajectory          510 – 570  ( 2 s)
# Scene 8 : Telemetry replay (faster)       570 – 720  ( 5 s)
# Scene 9 : Closing title card              720 – 780  ( 2 s)
# ─────────────────────────────────────────────────────────────
SCENE = {
    "title_start"   : 0,
    "title_end"     : 60,
    "axes_start"    : 60,
    "axes_end"      : 120,
    "launch_start"  : 120,
    "launch_end"    : 330,
    "height_start"  : 330,
    "height_end"    : 390,
    "range_start"   : 390,
    "range_end"     : 450,
    "tof_start"     : 450,
    "tof_end"       : 510,
    "traj_start"    : 510,
    "traj_end"      : 570,
    "replay_start"  : 570,
    "replay_end"    : 720,
    "closing_start" : 720,
    "closing_end"   : 780,
}
TOTAL_FRAMES = SCENE["closing_end"]

def ease_in_out(t):
    """Smooth cubic ease-in-out: t in [0,1] → [0,1]."""
    return t * t * (3 - 2 * t)

def ease_out(t):
    return 1 - (1 - t)**3

def clamp(val, lo=0.0, hi=1.0):
    return max(lo, min(hi, val))

def scene_progress(frame, start, end):
    if frame <= start: return 0.0
    if frame >= end:   return 1.0
    return ease_in_out((frame - start) / (end - start))

# ─────────────────────────────────────────────────────────────
# FIGURE SETUP
# ─────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(frame_width_in, frame_height_in), facecolor=CLR_BG)

# Layout: left physics canvas (70%) | right info panel (30%)
gs = gridspec.GridSpec(
    1, 2,
    left=0.03, right=0.97,
    bottom=0.05, top=0.93,
    wspace=0.04,
    width_ratios=[0.70, 0.30],
)
ax_physics = fig.add_subplot(gs[0, 0])   # main simulation view
ax_panel   = fig.add_subplot(gs[0, 1])   # info / equation panel

# ── Physics axes styling ─────────────────────────────────────
ax_physics.set_facecolor(CLR_BG)
for spine in ax_physics.spines.values():
    spine.set_color(CLR_PANEL_EDGE)
    spine.set_linewidth(1.2)
ax_physics.tick_params(colors=CLR_TEXT_DIM, labelsize=7)
ax_physics.set_xlim(*WORLD_X)
ax_physics.set_ylim(*WORLD_Y)
ax_physics.set_aspect("equal", adjustable="box")
ax_physics.grid(True, color=CLR_GRID, linewidth=0.5, linestyle="--", alpha=0.7)
ax_physics.set_xlabel("Horizontal Distance  x (m)", color=CLR_TEXT_MID,
                       fontsize=8, fontfamily=FONT_BODY, labelpad=4)
ax_physics.set_ylabel("Vertical Height  y (m)", color=CLR_TEXT_MID,
                       fontsize=8, fontfamily=FONT_BODY, labelpad=4)

# ── Panel axes styling ───────────────────────────────────────
ax_panel.set_facecolor(CLR_BG)
ax_panel.set_xlim(0, 1)
ax_panel.set_ylim(0, 1)
ax_panel.axis("off")

# ─────────────────────────────────────────────────────────────
# HELPER — draw a rounded card on ax_panel
# ─────────────────────────────────────────────────────────────
def draw_card(ax, x, y, w, h, title, color_accent, lines,
              title_size=7.5, body_size=7.0, alpha_box=0.92):
    """Draw a labelled card at (x,y) with width w and height h (axes coords)."""
    box = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.01",
        linewidth=1.2,
        edgecolor=color_accent,
        facecolor=CLR_PANEL,
        alpha=alpha_box,
        zorder=10,
        transform=ax.transAxes,
    )
    ax.add_patch(box)
    # accent bar at top
    bar = FancyBboxPatch(
        (x, y + h - 0.028), w, 0.028,
        boxstyle="round,pad=0.005",
        linewidth=0,
        facecolor=color_accent,
        alpha=0.9,
        zorder=11,
        transform=ax.transAxes,
    )
    ax.add_patch(bar)
    ax.text(x + w/2, y + h - 0.014, title,
            color=CLR_BG, fontsize=title_size, fontweight="bold",
            ha="center", va="center", fontfamily=FONT_BODY,
            zorder=12, transform=ax.transAxes)
    line_h = (h - 0.038) / max(len(lines), 1)
    for i, (label, val) in enumerate(lines):
        yy = y + h - 0.044 - i * line_h - line_h * 0.3
        ax.text(x + 0.015, yy, label, color=CLR_TEXT_MID,
                fontsize=body_size - 0.5, ha="left", va="center",
                fontfamily=FONT_BODY, zorder=12, transform=ax.transAxes)
        ax.text(x + w - 0.015, yy, val, color=CLR_TEXT_HI,
                fontsize=body_size, ha="right", va="center",
                fontweight="bold", fontfamily=FONT_BODY, zorder=12,
                transform=ax.transAxes)

# ─────────────────────────────────────────────────────────────
# WATERMARK
# ─────────────────────────────────────────────────────────────
watermark_text = "© Mugambi Ndwiga / @craftsandengineering"
fig.text(
    0.985, 0.012,
    watermark_text,
    color=CLR_WATERMARK, alpha=0.38,
    fontsize=5.5, ha="right", va="bottom",
    fontfamily=FONT_BODY,
    zorder=999,
)

# ─────────────────────────────────────────────────────────────
# PERSISTENT ARTISTS (created once, updated each frame)
# ─────────────────────────────────────────────────────────────

# Full arc (revealed gradually)
arc_line, = ax_physics.plot([], [], color=CLR_TRAJECTORY,
                             linewidth=2.2, alpha=0.85, zorder=4)

# Projectile circle
proj_circle = plt.Circle((0, 0), projectile_radius,
                          color=CLR_TRAJECTORY, zorder=8)
proj_circle.set_visible(False)
ax_physics.add_patch(proj_circle)

# Velocity vector (Vx)
vx_arrow = ax_physics.annotate(
    "", xy=(0, 0), xytext=(0, 0),
    arrowprops=dict(arrowstyle="-|>", color=CLR_VELOCITY,
                    lw=1.8, mutation_scale=12),
    zorder=7
)
# Velocity vector (Vy)
vy_arrow = ax_physics.annotate(
    "", xy=(0, 0), xytext=(0, 0),
    arrowprops=dict(arrowstyle="-|>", color=CLR_HEIGHT,
                    lw=1.8, mutation_scale=12),
    zorder=7
)
# Resultant velocity vector
vr_arrow = ax_physics.annotate(
    "", xy=(0, 0), xytext=(0, 0),
    arrowprops=dict(arrowstyle="-|>", color=CLR_ANGLE_ARC,
                    lw=2.0, mutation_scale=13),
    zorder=7
)

# Gravity arrow (downward)
grav_arrow = ax_physics.annotate(
    "", xy=(0, 0), xytext=(0, 0),
    arrowprops=dict(arrowstyle="-|>", color=CLR_ACCEL,
                    lw=1.6, mutation_scale=11),
    zorder=7
)

# Height dashed line
height_line, = ax_physics.plot([], [], "--", color=CLR_HEIGHT,
                                linewidth=1.4, alpha=0.8, zorder=3)
height_label = ax_physics.text(0, 0, "", color=CLR_HEIGHT,
                                fontsize=8.5, fontfamily=FONT_MATH,
                                zorder=9, ha="left")

# Range arrow annotation
range_line, = ax_physics.plot([], [], color=CLR_RANGE,
                               linewidth=2.0, alpha=0.85, zorder=3)
range_label = ax_physics.text(0, 0, "", color=CLR_RANGE,
                               fontsize=8.5, fontfamily=FONT_MATH,
                               zorder=9, ha="center")

# Angle arc
angle_arc = Arc((0, 0), 3.0, 3.0, angle=0,
                theta1=0, theta2=launch_angle_deg,
                color=CLR_ANGLE_ARC, linewidth=1.6, zorder=6)
angle_arc.set_visible(False)
ax_physics.add_patch(angle_arc)
angle_label = ax_physics.text(0, 0, "", color=CLR_ANGLE_ARC,
                               fontsize=9, fontfamily=FONT_MATH, zorder=9)

# Launch velocity arrow (shown at start)
launch_arrow = ax_physics.annotate(
    "", xy=(0, 0), xytext=(0, 0),
    arrowprops=dict(arrowstyle="-|>", color=CLR_ANGLE_ARC,
                    lw=2.2, mutation_scale=14),
    zorder=7
)

# Telemetry text (live values on physics canvas)
telem_speed = ax_physics.text(
    0, 0, "", color=CLR_TEXT_HI, fontsize=7.5,
    fontfamily=FONT_BODY, zorder=9,
    bbox=dict(boxstyle="round,pad=0.3", fc=CLR_PANEL, ec=CLR_PANEL_EDGE,
              alpha=0.88, lw=0.8)
)
telem_speed.set_visible(False)

# Title on physics canvas (shown in early frames)
phys_title = ax_physics.text(
    0.5, 0.96, "", color=CLR_TEXT_HI,
    fontsize=14, fontweight="bold",
    fontfamily=FONT_TITLE, ha="center", va="top",
    transform=ax_physics.transAxes, zorder=20,
    path_effects=[pe.withStroke(linewidth=3, foreground=CLR_BG)]
)
phys_subtitle = ax_physics.text(
    0.5, 0.88, "", color=CLR_TEXT_MID,
    fontsize=8, fontfamily=FONT_BODY,
    ha="center", va="top",
    transform=ax_physics.transAxes, zorder=20,
)

# ─────────────────────────────────────────────────────────────
# PANEL — persistent card backgrounds (drawn once, alpha toggled)
# ─────────────────────────────────────────────────────────────
# We'll draw the panel contents each frame as text annotations
# (simpler than managing many patch alphas)
panel_artists = []   # cleared and redrawn each frame


# ─────────────────────────────────────────────────────────────
# SCENE OVERLAY  (title card, equation reveals, closing)
# These use fig.text and figure patches for full-screen effects.
# ─────────────────────────────────────────────────────────────
overlay_bg = mpatches.Rectangle(
    (0, 0), 1, 1,
    transform=fig.transFigure,
    color=CLR_BG, alpha=0.0, zorder=50
)
fig.add_artist(overlay_bg)

overlay_title = fig.text(
    0.5, 0.56, "",
    color=CLR_TEXT_HI, fontsize=30, fontweight="bold",
    ha="center", va="center", fontfamily=FONT_TITLE,
    zorder=55, alpha=0.0,
)
overlay_subtitle = fig.text(
    0.5, 0.46, "",
    color=CLR_TEXT_MID, fontsize=11, ha="center", va="center",
    fontfamily=FONT_BODY, zorder=55, alpha=0.0,
)
overlay_hook = fig.text(
    0.5, 0.36, "",
    color=CLR_TRAJECTORY, fontsize=9.5, ha="center", va="center",
    fontfamily=FONT_BODY, style="italic", zorder=55, alpha=0.0,
)

# ─────────────────────────────────────────────────────────────
# TRAIL
# ─────────────────────────────────────────────────────────────
trail_xs = []
trail_ys = []
trail_line, = ax_physics.plot([], [], color=CLR_TRAJECTORY,
                               linewidth=1.0, alpha=0.4, zorder=3,
                               linestyle=":")

# ─────────────────────────────────────────────────────────────
# PANEL DRAW HELPER
# ─────────────────────────────────────────────────────────────
def render_panel(frame, t_current):
    """Redraw the right-hand info panel according to the current frame."""
    # Clear old panel patches / texts added dynamically
    for art in panel_artists:
        art.remove()
    panel_artists.clear()

    def padd(artist):
        panel_artists.append(artist)
        return artist

    # Panel header
    padd(ax_panel.text(
        0.5, 0.97, "PROJECTILE MOTION",
        color=CLR_TRAJECTORY, fontsize=9.5, fontweight="bold",
        ha="center", va="top", fontfamily=FONT_TITLE,
        transform=ax_panel.transAxes, zorder=12,
    ))

    sep = mpatches.FancyBboxPatch(
        (0.0, 0.955), 1.0, 0.002,
        boxstyle="square,pad=0",
        facecolor=CLR_TRAJECTORY, alpha=0.6,
        transform=ax_panel.transAxes, zorder=12
    )
    ax_panel.add_patch(sep)
    padd(sep)

    # ── LAUNCH PARAMETERS card ───────────────────────────────
    card_alpha = clamp(scene_progress(frame, SCENE["axes_start"], SCENE["axes_end"]))
    if card_alpha > 0.02:
        box = FancyBboxPatch(
            (0.02, 0.78), 0.96, 0.165,
            boxstyle="round,pad=0.01",
            linewidth=1.1, edgecolor=CLR_ANGLE_ARC,
            facecolor=CLR_PANEL, alpha=card_alpha * 0.92,
            transform=ax_panel.transAxes, zorder=10
        )
        ax_panel.add_patch(box)
        padd(box)

        bar = FancyBboxPatch(
            (0.02, 0.918), 0.96, 0.027,
            boxstyle="square,pad=0",
            facecolor=CLR_ANGLE_ARC, alpha=card_alpha * 0.85,
            transform=ax_panel.transAxes, zorder=11
        )
        ax_panel.add_patch(bar)
        padd(bar)

        padd(ax_panel.text(0.5, 0.932, "LAUNCH PARAMETERS",
            color=CLR_BG, fontsize=6.8, fontweight="bold",
            ha="center", va="center", fontfamily=FONT_BODY,
            transform=ax_panel.transAxes, zorder=12,
            alpha=card_alpha))

        params = [
            ("θ  (launch angle)", f"{launch_angle_deg}°"),
            ("u  (initial speed)", f"{initial_speed:.1f} m/s"),
            ("g  (gravity)",       f"{gravity:.2f} m/s²"),
        ]
        for i, (lbl, val) in enumerate(params):
            yy = 0.895 - i * 0.038
            padd(ax_panel.text(0.06, yy, lbl,
                color=CLR_TEXT_MID, fontsize=6.8, ha="left", va="center",
                fontfamily=FONT_BODY, transform=ax_panel.transAxes,
                zorder=12, alpha=card_alpha))
            padd(ax_panel.text(0.94, yy, val,
                color=CLR_ANGLE_ARC, fontsize=6.8, ha="right", va="center",
                fontweight="bold", fontfamily=FONT_BODY,
                transform=ax_panel.transAxes, zorder=12, alpha=card_alpha))

    # ── TELEMETRY card (live values) ─────────────────────────
    telem_alpha = clamp(scene_progress(frame, SCENE["launch_start"],
                                        SCENE["launch_start"] + 20))
    if telem_alpha > 0.02 and frame < SCENE["closing_start"]:
        cx = x_pos(t_current)
        cy = y_pos(t_current)
        cur_vx = vx(t_current)
        cur_vy = vy(t_current)
        cur_spd = speed(t_current)
        ke_ratio = cur_spd**2 / initial_speed**2  # normalised KE

        box = FancyBboxPatch(
            (0.02, 0.545), 0.96, 0.225,
            boxstyle="round,pad=0.01",
            linewidth=1.1, edgecolor=CLR_VELOCITY,
            facecolor=CLR_PANEL, alpha=telem_alpha * 0.92,
            transform=ax_panel.transAxes, zorder=10
        )
        ax_panel.add_patch(box)
        padd(box)

        bar2 = FancyBboxPatch(
            (0.02, 0.742), 0.96, 0.027,
            boxstyle="square,pad=0",
            facecolor=CLR_VELOCITY, alpha=telem_alpha * 0.85,
            transform=ax_panel.transAxes, zorder=11
        )
        ax_panel.add_patch(bar2)
        padd(bar2)

        padd(ax_panel.text(0.5, 0.756, "LIVE TELEMETRY",
            color=CLR_BG, fontsize=6.8, fontweight="bold",
            ha="center", va="center", fontfamily=FONT_BODY,
            transform=ax_panel.transAxes, zorder=12, alpha=telem_alpha))

        telem_rows = [
            ("x (m)",    f"{cx:.1f}",       CLR_TEXT_HI),
            ("y (m)",    f"{cy:.1f}",       CLR_TEXT_HI),
            ("Vx (m/s)", f"{cur_vx:.2f}",   CLR_VELOCITY),
            ("Vy (m/s)", f"{cur_vy:.2f}",   CLR_HEIGHT),
            ("‖V‖ (m/s)",f"{cur_spd:.2f}",  CLR_ANGLE_ARC),
            ("KE ratio", f"{ke_ratio:.2f}", CLR_RANGE),
        ]
        for i, (lbl, val, col) in enumerate(telem_rows):
            yy = 0.724 - i * 0.030
            padd(ax_panel.text(0.06, yy, lbl,
                color=CLR_TEXT_MID, fontsize=6.5, ha="left", va="center",
                fontfamily=FONT_BODY, transform=ax_panel.transAxes,
                zorder=12, alpha=telem_alpha))
            padd(ax_panel.text(0.94, yy, val,
                color=col, fontsize=6.8, ha="right", va="center",
                fontweight="bold", fontfamily=FONT_BODY,
                transform=ax_panel.transAxes, zorder=12, alpha=telem_alpha))

        # Mini KE bar
        bar_bg = FancyBboxPatch(
            (0.06, 0.548), 0.88, 0.014,
            boxstyle="square,pad=0", facecolor=CLR_PANEL_EDGE,
            transform=ax_panel.transAxes, zorder=12, alpha=telem_alpha
        )
        ax_panel.add_patch(bar_bg)
        padd(bar_bg)
        bar_fill = FancyBboxPatch(
            (0.06, 0.548), 0.88 * ke_ratio, 0.014,
            boxstyle="square,pad=0", facecolor=CLR_RANGE,
            transform=ax_panel.transAxes, zorder=13, alpha=telem_alpha
        )
        ax_panel.add_patch(bar_fill)
        padd(bar_fill)
        padd(ax_panel.text(0.5, 0.560, "Kinetic Energy",
            color=CLR_TEXT_DIM, fontsize=5.5, ha="center", va="center",
            fontfamily=FONT_BODY, transform=ax_panel.transAxes,
            zorder=14, alpha=telem_alpha))

    # ── KEY EQUATIONS cards ─────────────────────────────────
    eq_data = [
        ("TIME OF FLIGHT", CLR_TIME,
         r"$T = \dfrac{2u\sin\theta}{g}$",
         f"T = {time_of_flight:.2f} s",
         SCENE["tof_start"], 0.40),
        ("MAXIMUM HEIGHT", CLR_HEIGHT,
         r"$H = \dfrac{u^2\sin^2\!\theta}{2g}$",
         f"H = {max_height:.2f} m",
         SCENE["height_start"], 0.27),
        ("RANGE", CLR_RANGE,
         r"$R = \dfrac{u^2\sin 2\theta}{g}$",
         f"R = {range_total:.2f} m",
         SCENE["range_start"], 0.14),
    ]
    for (title_e, col_e, latex_e, val_e, reveal_frame, y_bot) in eq_data:
        eq_alpha = clamp(scene_progress(frame, reveal_frame, reveal_frame + 30))
        if eq_alpha > 0.02:
            eqbox = FancyBboxPatch(
                (0.02, y_bot), 0.96, 0.122,
                boxstyle="round,pad=0.01",
                linewidth=1.1, edgecolor=col_e,
                facecolor=CLR_PANEL, alpha=eq_alpha * 0.92,
                transform=ax_panel.transAxes, zorder=10
            )
            ax_panel.add_patch(eqbox)
            padd(eqbox)

            ebar = FancyBboxPatch(
                (0.02, y_bot + 0.094), 0.96, 0.027,
                boxstyle="square,pad=0",
                facecolor=col_e, alpha=eq_alpha * 0.85,
                transform=ax_panel.transAxes, zorder=11
            )
            ax_panel.add_patch(ebar)
            padd(ebar)

            padd(ax_panel.text(0.5, y_bot + 0.108, title_e,
                color=CLR_BG, fontsize=6.2, fontweight="bold",
                ha="center", va="center", fontfamily=FONT_BODY,
                transform=ax_panel.transAxes, zorder=12, alpha=eq_alpha))

            padd(ax_panel.text(0.5, y_bot + 0.063, latex_e,
                color=col_e, fontsize=9.5, ha="center", va="center",
                fontfamily=FONT_MATH, transform=ax_panel.transAxes,
                zorder=12, alpha=eq_alpha))

            padd(ax_panel.text(0.5, y_bot + 0.022, val_e,
                color=CLR_TEXT_HI, fontsize=7.0, ha="center", va="center",
                fontfamily=FONT_BODY, fontweight="bold",
                transform=ax_panel.transAxes, zorder=12, alpha=eq_alpha))

    # ── Trajectory equation (Scene 7) ───────────────────────
    tr_alpha = clamp(scene_progress(frame, SCENE["traj_start"],
                                     SCENE["traj_start" ] + 35))
    if tr_alpha > 0.02:
        tbox = FancyBboxPatch(
            (0.02, 0.01), 0.96, 0.125,
            boxstyle="round,pad=0.01",
            linewidth=1.1, edgecolor=CLR_VELOCITY,
            facecolor=CLR_PANEL, alpha=tr_alpha * 0.92,
            transform=ax_panel.transAxes, zorder=10
        )
        ax_panel.add_patch(tbox)
        padd(tbox)

        tbar = FancyBboxPatch(
            (0.02, 0.107), 0.96, 0.027,
            boxstyle="square,pad=0",
            facecolor=CLR_VELOCITY, alpha=tr_alpha * 0.85,
            transform=ax_panel.transAxes, zorder=11
        )
        ax_panel.add_patch(tbar)
        padd(tbar)

        padd(ax_panel.text(0.5, 0.121, "TRAJECTORY  y vs x",
            color=CLR_BG, fontsize=6.2, fontweight="bold",
            ha="center", va="center", fontfamily=FONT_BODY,
            transform=ax_panel.transAxes, zorder=12, alpha=tr_alpha))

        padd(ax_panel.text(0.5, 0.072,
            r"$y = x\tan\theta - \dfrac{gx^2}{2u^2\cos^2\!\theta}$",
            color=CLR_VELOCITY, fontsize=8.5, ha="center", va="center",
            fontfamily=FONT_MATH,
            transform=ax_panel.transAxes, zorder=12, alpha=tr_alpha))

        padd(ax_panel.text(0.5, 0.022,
            "Parabolic path in x-y plane",
            color=CLR_TEXT_MID, fontsize=6.0, ha="center", va="center",
            fontfamily=FONT_BODY, style="italic",
            transform=ax_panel.transAxes, zorder=12, alpha=tr_alpha))


# ─────────────────────────────────────────────────────────────
# MAIN UPDATE FUNCTION
# ─────────────────────────────────────────────────────────────
def update(frame):
    # ── SCENE 1: Hook title card ─────────────────────────────
    if frame < SCENE["title_end"]:
        prog = scene_progress(frame, 0, 45)
        fade_out = 1.0 if frame < 45 else 1.0 - ease_in_out((frame - 45) / 15)
        overlay_bg.set_alpha(fade_out * 0.97)
        overlay_title.set_text("PROJECTILE MOTION")
        overlay_title.set_alpha(prog * fade_out)
        overlay_subtitle.set_text(
            f"Launch at θ = {launch_angle_deg}°, u = {initial_speed} m/s")
        overlay_subtitle.set_alpha(max(0, prog - 0.3) * fade_out)
        overlay_hook.set_text(
            "What parabola does a thrown object trace through the air?")
        overlay_hook.set_alpha(max(0, prog - 0.5) * fade_out)
        phys_title.set_text("")
        arc_line.set_data([], [])
        proj_circle.set_visible(False)
        angle_arc.set_visible(False)
        angle_label.set_text("")
        height_line.set_data([], [])
        height_label.set_text("")
        range_line.set_data([], [])
        range_label.set_text("")
        trail_line.set_data([], [])
        telem_speed.set_visible(False)
        _hide_arrows()
        render_panel(frame, 0)
        return

    overlay_bg.set_alpha(0.0)
    overlay_title.set_alpha(0.0)
    overlay_subtitle.set_alpha(0.0)
    overlay_hook.set_alpha(0.0)

    # ── SCENE 2: Axes + angle reveal ─────────────────────────
    if frame < SCENE["axes_end"]:
        prog = scene_progress(frame, SCENE["axes_start"], SCENE["axes_end"])
        ax_scale = ease_out(prog)
        # Draw launch arrow growing in
        arrow_len = 6.0 * ax_scale
        ax_tip = (arrow_len * cos_theta, arrow_len * sin_theta)
        launch_arrow.set_position((0, 0))
        launch_arrow.xy = ax_tip
        launch_arrow.xyann = (0, 0)

        angle_arc.set_visible(prog > 0.4)
        if prog > 0.4:
            angle_arc._theta2 = launch_angle_deg * min(1.0, (prog - 0.4) / 0.6)
        angle_label.set_text(f"θ={launch_angle_deg}°")
        angle_label.set_position((2.2, 0.6))
        angle_label.set_alpha(max(0, prog - 0.5))

        phys_title.set_text("PROJECTILE MOTION")
        phys_subtitle.set_text(
            f"θ = {launch_angle_deg}°   |   u = {initial_speed} m/s   |   g = {gravity} m/s²")

        proj_circle.set_visible(True)
        proj_circle.set_center((0, 0))
        arc_line.set_data([], [])
        height_line.set_data([], [])
        range_line.set_data([], [])
        trail_line.set_data([], [])
        telem_speed.set_visible(False)
        _hide_arrows(keep_launch=True)
        render_panel(frame, 0)
        return

    # ── SCENES 3 + 8: Projectile flight ─────────────────────
    is_replay = (frame >= SCENE["replay_start"] and
                 frame < SCENE["replay_end"])

    if (SCENE["launch_start"] <= frame < SCENE["launch_end"]) or is_replay:
        if is_replay:
            flight_prog = scene_progress(
                frame, SCENE["replay_start"], SCENE["replay_end"])
            # replay faster: compress to 85% of range
            flight_prog = clamp(flight_prog * 1.2)
        else:
            flight_prog = scene_progress(
                frame, SCENE["launch_start"], SCENE["launch_end"])

        t_now = flight_prog * time_of_flight
        cx = x_pos(t_now)
        cy = max(0.0, y_pos(t_now))

        # Trail
        trail_xs.append(cx)
        trail_ys.append(cy)
        if len(trail_xs) > trail_length:
            trail_xs.pop(0)
            trail_ys.pop(0)
        trail_line.set_data(trail_xs, trail_ys)

        # Arc revealed up to current position
        n_reveal = max(2, int(flight_prog * len(t_dense)))
        arc_line.set_data(traj_x[:n_reveal], traj_y[:n_reveal])

        proj_circle.set_center((cx, cy))
        proj_circle.set_visible(True)

        # Velocity vectors (scaled for visibility)
        vscale = 0.45
        cur_vx = vx(t_now) * vscale
        cur_vy = vy(t_now) * vscale
        cur_spd = speed(t_now)

        vx_arrow.set_position((cx, cy))
        vx_arrow.xy = (cx + cur_vx, cy)
        vy_arrow.set_position((cx, cy))
        vy_arrow.xy = (cx, cy + cur_vy)
        vr_arrow.set_position((cx, cy))
        vr_arrow.xy = (cx + cur_vx, cy + cur_vy)
        grav_arrow.set_position((cx, cy + 1.2))
        grav_arrow.xy = (cx, cy - 0.4)

        # Live telemetry badge on canvas
        telem_speed.set_position((cx + 0.5, cy + 1.5))
        telem_speed.set_text(
            f" t = {t_now:.2f}s   ‖V‖ = {cur_spd:.1f} m/s ")
        telem_speed.set_visible(True)

        angle_arc.set_visible(False)
        angle_label.set_text("")
        launch_arrow.xy = (0, 0)
        launch_arrow.xyann = (0, 0)
        phys_title.set_text("")
        phys_subtitle.set_text("")
        render_panel(frame, t_now)
        return

    # ── SCENE 4: Height annotation ───────────────────────────
    if SCENE["height_start"] <= frame < SCENE["height_end"]:
        prog = scene_progress(frame, SCENE["height_start"], SCENE["height_end"])
        apex_x = range_total / 2
        apex_y = max_height
        hl = prog * apex_y
        height_line.set_data([apex_x, apex_x], [0, hl])
        height_label.set_position((apex_x + 0.4, hl / 2))
        height_label.set_text(f"H = {max_height:.1f} m")
        height_label.set_alpha(prog)
        arc_line.set_data(traj_x, traj_y)
        proj_circle.set_center((apex_x, apex_y))
        telem_speed.set_visible(False)
        _hide_arrows()
        trail_line.set_data([], [])
        range_line.set_data([], [])
        range_label.set_text("")
        render_panel(frame, time_of_flight / 2)
        return

    # ── SCENE 5: Range annotation ────────────────────────────
    if SCENE["range_start" ] <= frame < SCENE["range_end"]:
        prog = scene_progress(frame, SCENE["range_start"], SCENE["range_end"])
        rl = prog * range_total
        range_line.set_data([0, rl], [-max_height * 0.06, -max_height * 0.06])
        range_label.set_position((rl / 2, -max_height * 0.13))
        range_label.set_text(f"R = {range_total:.1f} m")
        range_label.set_alpha(prog)
        arc_line.set_data(traj_x, traj_y)
        proj_circle.set_visible(False)
        telem_speed.set_visible(False)
        _hide_arrows()
        trail_line.set_data([], [])
        render_panel(frame, time_of_flight)
        return

    # ── SCENE 6: Time-of-flight card ─────────────────────────
    if SCENE["tof_start"] <= frame < SCENE["tof_end"]:
        arc_line.set_data(traj_x, traj_y)
        proj_circle.set_visible(False)
        telem_speed.set_visible(False)
        _hide_arrows()
        trail_line.set_data([], [])
        render_panel(frame, time_of_flight)
        return

    # ── SCENE 7: Trajectory equation card ───────────────────
    if SCENE["traj_start"] <= frame < SCENE["traj_end"]:
        arc_line.set_data(traj_x, traj_y)
        proj_circle.set_visible(False)
        telem_speed.set_visible(False)
        _hide_arrows()
        trail_line.set_data([], [])
        render_panel(frame, time_of_flight)
        return

    # ── SCENE 9: Closing title card ──────────────────────────
    if frame >= SCENE["closing_start"]:
        prog = scene_progress(frame,
                              SCENE["closing_start" ],
                              SCENE["closing_start"] + 20)
        fade = prog

        overlay_bg.set_alpha(fade * 0.98)

        overlay_title.set_text("PROJECTILE MOTION")
        overlay_title.set_alpha(fade)

        overlay_subtitle.set_text("Made by Mugambi Ndwiga\n@craftsandengineering")
        overlay_subtitle.set_alpha(fade)

        overlay_hook.set_text(
            f"θ={launch_angle_deg}°  u={initial_speed}m/s  "
            f"T={time_of_flight:.2f}s  H={max_height:.1f}m  R={range_total:.1f}m")
        overlay_hook.set_alpha(fade * 0.8)

        arc_line.set_data(traj_x, traj_y)
        proj_circle.set_visible(False)
        telem_speed.set_visible(False)
        _hide_arrows()
        render_panel(frame, time_of_flight)
        return


def _hide_arrows(keep_launch=False):
    """Hide all dynamic arrows."""
    for arr in [vx_arrow, vy_arrow, vr_arrow, grav_arrow]:
        arr.set_position((0, 0))
        arr.xy = (0, 0)
    if not keep_launch:
        launch_arrow.set_position((0, 0))
        launch_arrow.xy = (0, 0)


# ─────────────────────────────────────────────────────────────
# RENDER
# ─────────────────────────────────────────────────────────────
print(f"  Rendering {TOTAL_FRAMES} frames at {video_fps} fps  …")
print(f"    Resolution: {int(frame_width_in * video_dpi)} × "
      f"{int(frame_height_in * video_dpi)} px")
print(f"    Output: {output_path}")

metadata = dict(
    title="Projectile Motion",
    artist="Mugambi Ndwiga / @craftsandengineering",
    comment=(
        "Educational animation — projectile motion physics. "
        "github.com/zombimann/Mathematical-video-animations-and-visualization"
    ),
)

writer = FFMpegWriter(
    fps=video_fps,
    metadata=metadata,
    extra_args=[
        "-vcodec", "libx264",
        "-crf", "23",
        "-preset", "slow",
        "-pix_fmt", "yuv420p",
        "-movflags", "+faststart",
    ],
)

os.makedirs(os.path.dirname(output_path), exist_ok=True)

with writer.saving(fig, output_path, dpi=video_dpi):
    for f in range(TOTAL_FRAMES):
        # Reset trail between scenes
        if f == SCENE["launch_start"] or f == SCENE["replay_start"]:
            trail_xs.clear()
            trail_ys.clear()
        update(f)
        writer.grab_frame()
        if f % 60 == 0:
            pct = 100 * f / TOTAL_FRAMES
            print(f"    [{pct:5.1f}%]  frame {f}/{TOTAL_FRAMES}", flush=True)

plt.close(fig)

file_size_mb = os.path.getsize(output_path) / 1e6
print(f"\n  Done!  File size: {file_size_mb:.2f} MB")
print(f"    Saved to: {output_path}")

# Display and Download
mp4 = open(output_path,'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
display(HTML(f"""
<video width="800" controls>
      <source src="{data_url}" type="video/mp4">
</video>
"""))
files.download(output_path)

  Rendering 780 frames at 30 fps  …
    Resolution: 1584 × 891 px
    Output: ./projectile_motion.mp4
    [  0.0%]  frame 0/780
    [  7.7%]  frame 60/780
    [ 15.4%]  frame 120/780
    [ 23.1%]  frame 180/780
    [ 30.8%]  frame 240/780
    [ 38.5%]  frame 300/780
    [ 46.2%]  frame 360/780
    [ 53.8%]  frame 420/780
    [ 61.5%]  frame 480/780
    [ 69.2%]  frame 540/780
    [ 76.9%]  frame 600/780
    [ 84.6%]  frame 660/780
    [ 92.3%]  frame 720/780

  Done!  File size: 0.53 MB
    Saved to: ./projectile_motion.mp4


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>